# Section 1 — Notebook Overview & Architecture Design Rationale

**Task 2: MLP Architecture Development**  
**Project 18:** Surface Water Quality Prediction via Gradient Boosting & Deep Neural Networks  
**Target:** CCME WQI (`ccme_values`) — continuous regression, range \[0, 100\]  
**Input:** 53-dimensional feature vector (36 numerical + 17 one-hot categorical)  

---

## Design Rationale

Phase 1 established that orthophosphate drives >88% of CCME WQI variance. The MLP must learn:
1. **Non-linear saturation curves** — log-transformed nutrients interact non-linearly with the WQI formula.
2. **Multi-scale interactions** — chemical ratios, lags, and cyclical temporal encodings operate at different scales.
3. **Regularized generalization** — 2.8M training rows risk confident overfitting without BatchNorm + Dropout.

## Architecture Family Explored

| Experiment | Hidden Layers | Neurons/Layer | Dropout | BatchNorm | Purpose |
|---|---|---|---|---|---|
| `MLP_Tiny` | 2 | 128 | 0.0 | No | Minimum viable baseline |
| `MLP_Small` | 3 | 256 | 0.2 | No | Underfitting reference |
| `MLP_Base` | 4 | 512→256→128→64 | 0.3 | Yes | **Primary architecture** |
| `MLP_Wide` | 3 | 1024→512→256 | 0.3 | Yes | Width ablation |
| `MLP_Deep` | 6 | 512 (uniform) | 0.3 | Yes | Depth ablation |
| `MLP_ResNet` | 4+skip | 256 (bottleneck) | 0.2 | Yes | Residual connection ablation |

# Section 2 — Environment Setup & Imports

In [ ]:
import os
import sys
import json
import math
import time
import warnings
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Any
from collections import OrderedDict
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.alpha'] = 0.35
sns.set_palette('Set2')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version  : {torch.__version__}')
print(f'Computation device: {DEVICE}')
print(f'Random seed      : {SEED}')

# ── Path Resolution ──────────────────────────────────────────────────────────
current_dir = Path(os.path.abspath(''))
if (current_dir / 'Phase_1_Machine_Learning').exists():
    ROOT = current_dir
elif (current_dir.parent / 'Phase_1_Machine_Learning').exists():
    ROOT = current_dir.parent
elif (current_dir.parent.parent / 'Phase_1_Machine_Learning').exists():
    ROOT = current_dir.parent.parent
else:
    ROOT = current_dir

DATA_DIR   = ROOT / 'Phase_1_Machine_Learning' / 'data' / 'GeneratedOutput' / '02.Preprocessing&FeatureSelection'
PHASE2_DIR = ROOT / 'Phase_2_Deep_Learning'
ARTIFACTS  = PHASE2_DIR / 'artifacts'
MODELS_DIR = PHASE2_DIR / 'models'
FIGURES_DIR= PHASE2_DIR / 'figures'

for d in [ARTIFACTS, MODELS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'\nProject root    : {ROOT}')
print(f'Data directory  : {DATA_DIR}')
print(f'Models directory: {MODELS_DIR}')
print(f'Figures dir     : {FIGURES_DIR}')

# Section 3 — Load Phase 1 Preprocessed Data & Feature Config

In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ── Load splits ──────────────────────────────────────────────────────────────
print('Loading Phase 1 partitioned data...')
train_df = pd.read_parquet(DATA_DIR / 'train.parquet')
val_df   = pd.read_parquet(DATA_DIR / 'val.parquet')
test_df  = pd.read_parquet(DATA_DIR / 'test.parquet')

print(f'Train : {train_df.shape}')
print(f'Val   : {val_df.shape}')
print(f'Test  : {test_df.shape}')

# ── Feature config ───────────────────────────────────────────────────────────
with open(PHASE2_DIR / 'configs' / 'feature_config.json') as f:
    cfg = json.load(f)

TARGET               = cfg['target_variable']          # 'ccme_values'
NUMERICAL_FEATURES   = cfg['numerical_features']       # 36 features
CATEGORICAL_FEATURES = cfg['categorical_features']     # 2 raw cats
ONE_HOT_FEATURES     = cfg['one_hot_feature_names']    # 17 OHE cols
ALL_FEATURES         = cfg['all_feature_names']        # 53 total
INPUT_DIM            = cfg['total_input_dimension']    # 53
BATCH_SIZE           = cfg['batch_size']               # 1024

print(f'\nTarget            : {TARGET}')
print(f'Numerical features: {len(NUMERICAL_FEATURES)}')
print(f'OHE categories    : {len(ONE_HOT_FEATURES)}')
print(f'Total input dim   : {INPUT_DIM}')
print(f'Batch size        : {BATCH_SIZE}')

# Section 4 — Build PyTorch Tensors via Saved Transformers

In [ ]:
# Load artifacts fitted in notebook 01 (train-only fit — no leakage)
scaler  = joblib.load(ARTIFACTS / 'numerical_scaler.joblib')
encoder = joblib.load(ARTIFACTS / 'categorical_encoder.joblib')

print('Transformers loaded successfully:')
print(f'  Scaler type   : {type(scaler).__name__}')
print(f'  Encoder type  : {type(encoder).__name__}')

# ── Transform ────────────────────────────────────────────────────────────────
def prepare_tensors(
    df: pd.DataFrame,
    scaler: StandardScaler,
    encoder: OneHotEncoder,
    numerical_features: List[str],
    categorical_features: List[str],
    target: str,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Transform a split DataFrame into float32 PyTorch tensors."""
    X_num = scaler.transform(df[numerical_features].astype(np.float32))
    X_cat = encoder.transform(df[categorical_features])
    X_all = np.hstack([X_num, X_cat]).astype(np.float32)
    y     = df[target].values.astype(np.float32).reshape(-1, 1)

    return (
        torch.from_numpy(X_num.astype(np.float32)),
        torch.from_numpy(X_cat.astype(np.float32)),
        torch.from_numpy(X_all),
        torch.from_numpy(y),
    )

print('\nBuilding train tensors...')
x_train_num, x_train_cat, x_train_all, y_train = prepare_tensors(
    train_df, scaler, encoder, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TARGET)

print('Building val tensors...')
x_val_num, x_val_cat, x_val_all, y_val = prepare_tensors(
    val_df, scaler, encoder, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TARGET)

print('Building test tensors...')
x_test_num, x_test_cat, x_test_all, y_test = prepare_tensors(
    test_df, scaler, encoder, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TARGET)

print(f'\nTrain X shape : {x_train_all.shape}')
print(f'Train y shape : {y_train.shape}')
print(f'Val   X shape : {x_val_all.shape}')
print(f'Test  X shape : {x_test_all.shape}')

# Sanity checks
assert not torch.isnan(x_train_all).any(), 'NaN in train features'
assert not torch.isinf(x_train_all).any(), 'Inf in train features'
assert not torch.isnan(y_train).any(), 'NaN in train targets'
assert x_train_all.shape[1] == INPUT_DIM
print('All tensor sanity checks passed.')

# Section 5 — WaterQualityDataset & DataLoader

In [ ]:
class WaterQualityDataset(Dataset):
    """
    PyTorch Dataset for the CCME WQI regression task.

    Stores pre-computed float32 tensors for fast __getitem__ access.
    Supports two return modes:
      - 'combined'  : (X_all, y)           <- default, used by MLP
      - 'separated' : (X_num, X_cat, y)    <- for embedding-aware models
    """

    def __init__(
        self,
        x_numerical: torch.Tensor,
        x_categorical: torch.Tensor,
        x_combined: torch.Tensor,
        y: torch.Tensor,
        return_mode: str = 'combined',
    ):
        assert len(x_numerical) == len(y), 'Row count mismatch'
        self.x_num  = x_numerical
        self.x_cat  = x_categorical
        self.x_all  = x_combined
        self.y      = y
        self.return_mode = return_mode

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx):
        if self.return_mode == 'separated':
            return self.x_num[idx], self.x_cat[idx], self.y[idx]
        return self.x_all[idx], self.y[idx]

    def feature_dim(self) -> int:
        return self.x_all.shape[1]

    def num_dim(self) -> int:
        return self.x_num.shape[1]

    def cat_dim(self) -> int:
        return self.x_cat.shape[1]


# ── Instantiate datasets ──────────────────────────────────────────────────────
train_dataset = WaterQualityDataset(x_train_num, x_train_cat, x_train_all, y_train)
val_dataset   = WaterQualityDataset(x_val_num,   x_val_cat,   x_val_all,   y_val)
test_dataset  = WaterQualityDataset(x_test_num,  x_test_cat,  x_test_all,  y_test)

# ── DataLoaders ───────────────────────────────────────────────────────────────
NUM_WORKERS = 0   # 0 for macOS compatibility; set 4 on Linux GPU servers
PIN_MEMORY  = DEVICE.type == 'cuda'

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

print(f'Train dataset   : {len(train_dataset):>10,} rows | {len(train_loader):>5,} batches')
print(f'Val   dataset   : {len(val_dataset):>10,} rows | {len(val_loader):>5,} batches')
print(f'Test  dataset   : {len(test_dataset):>10,} rows | {len(test_loader):>5,} batches')
print(f'Feature dim     : {train_dataset.feature_dim()} (num={train_dataset.num_dim()}, cat={train_dataset.cat_dim()})')

# Quick shape verification
sample_x, sample_y = next(iter(train_loader))
print(f'\nSample batch X  : {tuple(sample_x.shape)}')
print(f'Sample batch y  : {tuple(sample_y.shape)}')
assert sample_x.shape == (BATCH_SIZE, INPUT_DIM)
assert sample_y.shape == (BATCH_SIZE, 1)
print('DataLoader sanity check: PASSED')

# Section 6 — MLPBlock: Reusable Building Block

Every hidden layer follows the **Pre-Activation pattern**:
```
Linear → BatchNorm → ReLU → Dropout
```
Pre-activation (BN before activation) is preferred for regression because:
- Gradient magnitudes remain bounded through deep stacks.
- BN normalizes pre-activation values, preventing dead ReLU neurons.
- Empirically outperforms Post-BN (BN after ReLU) on tabular regression.

In [ ]:
class MLPBlock(nn.Module):
    """
    Single MLP hidden layer block: Linear → [BatchNorm] → ReLU → [Dropout].

    Args:
        in_features   : Input dimension.
        out_features  : Output dimension.
        dropout_rate  : Dropout probability (0.0 = disabled).
        use_batch_norm: Whether to apply BatchNorm1d before ReLU.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        dropout_rate: float = 0.3,
        use_batch_norm: bool = True,
    ):
        super().__init__()
        layers = [nn.Linear(in_features, out_features)]
        if use_batch_norm:
            layers.append(nn.BatchNorm1d(out_features))
        layers.append(nn.ReLU(inplace=True))
        if dropout_rate > 0.0:
            layers.append(nn.Dropout(p=dropout_rate))
        self.block = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


# Quick unit test
test_block = MLPBlock(53, 256, dropout_rate=0.3, use_batch_norm=True)
test_block.eval()
dummy = torch.randn(16, 53)
out   = test_block(dummy)
assert out.shape == (16, 256), f'Expected (16, 256), got {out.shape}'
print('MLPBlock unit test: PASSED')
print(f'Block structure:\n{test_block}')

# Section 7 — WaterQualityMLP: Primary Architecture

## Architecture Specification

```
Input (53) → Block(53→512) → Block(512→256) → Block(256→128) → Block(128→64) → Linear(64→1)
             [BN+ReLU+DO]    [BN+ReLU+DO]     [BN+ReLU+DO]     [BN+ReLU+DO]   [raw logit]
```

**Funnel topology** (narrowing widths): captures coarse patterns in early layers,
refines into compact representations for regression output.

**Output layer**: single `Linear(64→1)` with **no activation** — raw prediction
covers the full real line. The CCME WQI is bounded \[0, 100\] by construction;
post-processing clips are applied during evaluation only.

In [ ]:
@dataclass
class MLPConfig:
    """
    Hyperparameter configuration for WaterQualityMLP.
    Stored as a dataclass for reproducible experiment tracking.
    """
    input_dim    : int   = 53
    hidden_dims  : List[int] = field(default_factory=lambda: [512, 256, 128, 64])
    dropout_rate : float = 0.30
    use_batch_norm: bool = True
    output_dim   : int   = 1
    name         : str   = 'MLP_Base'

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    @property
    def total_params(self) -> int:
        """Estimate total parameter count (approximate)."""
        dims = [self.input_dim] + self.hidden_dims + [self.output_dim]
        total = 0
        for i in range(len(dims) - 1):
            # weights + biases
            total += dims[i] * dims[i+1] + dims[i+1]
            # BatchNorm: 2 * out_features (gamma, beta)
            if self.use_batch_norm and i < len(dims) - 2:
                total += 2 * dims[i+1]
        return total


class WaterQualityMLP(nn.Module):
    """
    Fully-Connected Multi-Layer Perceptron for CCME WQI regression.

    Architecture:
        Input  →  [MLPBlock] × N  →  Linear(last_dim → 1)  →  scalar prediction

    Key design choices:
        - Funnel topology (decreasing hidden widths) for progressive abstraction.
        - BatchNorm1d before ReLU prevents internal covariate shift.
        - Dropout applied after each activation for regularization.
        - Raw linear output — no sigmoid/tanh — covers unbounded residuals.
        - Kaiming (He) weight initialization for ReLU-compatible convergence.

    Args:
        config: MLPConfig dataclass holding all hyperparameters.
    """

    def __init__(self, config: MLPConfig):
        super().__init__()
        self.config = config

        # ── Hidden blocks ────────────────────────────────────────────────────
        dims = [config.input_dim] + config.hidden_dims
        self.hidden_layers = nn.ModuleList([
            MLPBlock(
                in_features    = dims[i],
                out_features   = dims[i + 1],
                dropout_rate   = config.dropout_rate,
                use_batch_norm = config.use_batch_norm,
            )
            for i in range(len(dims) - 1)
        ])

        # ── Regression output head ───────────────────────────────────────────
        self.output_layer = nn.Linear(config.hidden_dims[-1], config.output_dim)

        # ── Weight initialization ────────────────────────────────────────────
        self._initialize_weights()

    def _initialize_weights(self):
        """Kaiming He initialization for ReLU networks."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.BatchNorm1d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass: x (B, 53) → predictions (B, 1).
        """
        for layer in self.hidden_layers:
            x = layer(x)
        return self.output_layer(x)

    def predict_clipped(self, x: torch.Tensor) -> torch.Tensor:
        """Inference-time forward pass with CCME [0, 100] clipping."""
        with torch.no_grad():
            return self.forward(x).clamp(0.0, 100.0)

    def count_parameters(self) -> Dict[str, int]:
        """Return trainable and total parameter counts."""
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {'total': total, 'trainable': trainable}

    def architecture_summary(self) -> str:
        """Human-readable architecture summary string."""
        dims = [self.config.input_dim] + self.config.hidden_dims + [self.config.output_dim]
        flow = ' → '.join([str(d) for d in dims])
        p    = self.count_parameters()
        return (
            f"Architecture : {self.config.name}\n"
            f"Layer flow   : {flow}\n"
            f"BatchNorm    : {self.config.use_batch_norm}\n"
            f"Dropout      : {self.config.dropout_rate}\n"
            f"Parameters   : {p['trainable']:,} trainable / {p['total']:,} total"
        )


# ── Instantiate primary model ────────────────────────────────────────────────
base_config = MLPConfig(
    input_dim     = INPUT_DIM,          # 53
    hidden_dims   = [512, 256, 128, 64],
    dropout_rate  = 0.30,
    use_batch_norm= True,
    output_dim    = 1,
    name          = 'MLP_Base',
)

model = WaterQualityMLP(base_config).to(DEVICE)
print(model.architecture_summary())
print(f'\nFull model structure:')
print(model)

# Section 8 — Architecture Forward-Pass Verification

In [ ]:
model.eval()

# ── Verify on a live DataLoader batch ───────────────────────────────────────
batch_x, batch_y = next(iter(train_loader))
batch_x = batch_x.to(DEVICE)

with torch.no_grad():
    preds = model(batch_x)

print('Forward-pass verification:')
print(f'  Input shape    : {tuple(batch_x.shape)}')
print(f'  Output shape   : {tuple(preds.shape)}')
print(f'  Output dtype   : {preds.dtype}')
print(f'  Output mean    : {preds.mean().item():.4f}')
print(f'  Output std     : {preds.std().item():.4f}')
print(f'  Output min     : {preds.min().item():.4f}')
print(f'  Output max     : {preds.max().item():.4f}')
print(f'  NaN in output  : {torch.isnan(preds).any().item()}')
print(f'  Inf in output  : {torch.isinf(preds).any().item()}')

assert preds.shape == (BATCH_SIZE, 1), f'Unexpected output shape: {preds.shape}'
assert not torch.isnan(preds).any(), 'NaN detected in model output'
assert not torch.isinf(preds).any(), 'Inf detected in model output'

# ── Memory footprint ─────────────────────────────────────────────────────────
param_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
print(f'\nModel memory footprint: {param_mb:.3f} MB')
print('Forward-pass verification: PASSED')

# Section 9 — Architecture Experiment Variants

Six architectures spanning a systematic grid:
- **Depth** axis: 2, 3, 4, 6 hidden layers
- **Width** axis: 128, 256, 512, 1024 neurons per layer
- **Regularization**: dropout 0.0, 0.2, 0.3; BatchNorm on/off
- **Topology**: monotone funnel vs flat uniform vs residual skip

In [ ]:
# ── Define all experiment configurations ─────────────────────────────────────
EXPERIMENT_CONFIGS: List[MLPConfig] = [
    # Minimum viable baseline — no regularization, shallow
    MLPConfig(
        input_dim=INPUT_DIM, hidden_dims=[128, 128],
        dropout_rate=0.0, use_batch_norm=False,
        name='MLP_Tiny',
    ),
    # 3-layer moderate with no BN — underfitting reference
    MLPConfig(
        input_dim=INPUT_DIM, hidden_dims=[256, 256, 256],
        dropout_rate=0.2, use_batch_norm=False,
        name='MLP_Small',
    ),
    # PRIMARY: 4-layer funnel with BN + Dropout
    MLPConfig(
        input_dim=INPUT_DIM, hidden_dims=[512, 256, 128, 64],
        dropout_rate=0.3, use_batch_norm=True,
        name='MLP_Base',
    ),
    # Width ablation — same depth, 2× wider first layer
    MLPConfig(
        input_dim=INPUT_DIM, hidden_dims=[1024, 512, 256],
        dropout_rate=0.3, use_batch_norm=True,
        name='MLP_Wide',
    ),
    # Depth ablation — deeper uniform width
    MLPConfig(
        input_dim=INPUT_DIM, hidden_dims=[512, 512, 512, 512, 256, 128],
        dropout_rate=0.3, use_batch_norm=True,
        name='MLP_Deep',
    ),
    # High dropout regularization experiment
    MLPConfig(
        input_dim=INPUT_DIM, hidden_dims=[512, 256, 128, 64],
        dropout_rate=0.5, use_batch_norm=True,
        name='MLP_Base_DO50',
    ),
]

# ── Build models and collect stats ───────────────────────────────────────────
architecture_registry: Dict[str, WaterQualityMLP] = {}
architecture_stats: List[Dict] = []

print(f'{"Architecture":<18} {"Hidden Dims":<30} {"BN":<5} {"DO":<6} {"Params":>12}')
print('─' * 75)

for cfg_exp in EXPERIMENT_CONFIGS:
    m = WaterQualityMLP(cfg_exp).to(DEVICE)
    p = m.count_parameters()
    architecture_registry[cfg_exp.name] = m
    architecture_stats.append({
        'name'         : cfg_exp.name,
        'hidden_dims'  : str(cfg_exp.hidden_dims),
        'depth'        : len(cfg_exp.hidden_dims),
        'max_width'    : max(cfg_exp.hidden_dims),
        'batch_norm'   : cfg_exp.use_batch_norm,
        'dropout'      : cfg_exp.dropout_rate,
        'trainable_params': p['trainable'],
        'total_params' : p['total'],
    })
    print(
        f"{cfg_exp.name:<18} {str(cfg_exp.hidden_dims):<30} "
        f"{'Y' if cfg_exp.use_batch_norm else 'N':<5} "
        f"{cfg_exp.dropout_rate:<6.1f} {p['trainable']:>12,}"
    )

stats_df = pd.DataFrame(architecture_stats)
print(f'\nArchitecture registry contains {len(architecture_registry)} models.')

# Section 10 — Residual MLP (ResNet-style Skip Connections)

In [ ]:
class ResidualBlock(nn.Module):
    """
    Residual block for tabular data:
        x → Linear → BN → ReLU → Dropout → Linear → BN → (+x) → ReLU

    When in_features ≠ out_features, a 1×1 projection (shortcut) aligns dimensions.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        dropout_rate: float = 0.2,
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(in_features, out_features),
            nn.BatchNorm1d(out_features),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(out_features, out_features),
            nn.BatchNorm1d(out_features),
        )

        # Projection shortcut when dimensions mismatch
        if in_features != out_features:
            self.shortcut = nn.Sequential(
                nn.Linear(in_features, out_features, bias=False),
                nn.BatchNorm1d(out_features),
            )
        else:
            self.shortcut = nn.Identity()

        self.activation = nn.ReLU(inplace=True)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.shortcut(x)
        out      = self.block(x)
        return self.activation(out + residual)


class ResidualMLP(nn.Module):
    """
    MLP with residual skip connections for CCME WQI regression.

    Architecture:
        Input Projection → [ResidualBlock × N] → Output Head

    Args:
        input_dim   : Input feature dimension (53).
        hidden_dim  : Uniform hidden dimension within residual blocks.
        num_blocks  : Number of residual blocks.
        dropout_rate: Dropout within each residual block.
    """

    def __init__(
        self,
        input_dim   : int   = 53,
        hidden_dim  : int   = 256,
        num_blocks  : int   = 4,
        dropout_rate: float = 0.2,
    ):
        super().__init__()
        self.name = 'MLP_ResNet'

        # Project input to hidden_dim
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

        # Stack residual blocks (uniform hidden_dim → no projection needed)
        self.residual_blocks = nn.ModuleList([
            ResidualBlock(hidden_dim, hidden_dim, dropout_rate)
            for _ in range(num_blocks)
        ])

        # Regression head
        self.output_head = nn.Linear(hidden_dim, 1)
        nn.init.kaiming_normal_(self.output_head.weight, nonlinearity='relu')
        nn.init.zeros_(self.output_head.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)
        for block in self.residual_blocks:
            x = block(x)
        return self.output_head(x)

    def count_parameters(self) -> Dict[str, int]:
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {'total': total, 'trainable': trainable}


# ── Instantiate & register ResidualMLP ───────────────────────────────────────
resnet_model = ResidualMLP(
    input_dim    = INPUT_DIM,
    hidden_dim   = 256,
    num_blocks   = 4,
    dropout_rate = 0.2,
).to(DEVICE)

architecture_registry['MLP_ResNet'] = resnet_model
resnet_params = resnet_model.count_parameters()

architecture_stats.append({
    'name'          : 'MLP_ResNet',
    'hidden_dims'   : '[256×4 blocks w/ skip]',
    'depth'         : 8,  # 2 linears per block × 4 blocks
    'max_width'     : 256,
    'batch_norm'    : True,
    'dropout'       : 0.2,
    'trainable_params': resnet_params['trainable'],
    'total_params'  : resnet_params['total'],
})

stats_df = pd.DataFrame(architecture_stats)

print('ResidualMLP summary:')
print(f'  Parameters: {resnet_params["trainable"]:,} trainable')
print(f'  Structure: Input({INPUT_DIM}) → Proj(256) → 4×ResBlock(256) → Linear(1)')
print(resnet_model)

# Forward pass check
resnet_model.eval()
with torch.no_grad():
    resnet_out = resnet_model(sample_x.to(DEVICE))
assert resnet_out.shape == (BATCH_SIZE, 1)
print('\nResidualMLP forward pass: PASSED')

# Section 11 — Architecture Comparison Summary Table

In [ ]:
print('=' * 90)
print('ARCHITECTURE EXPERIMENT SUMMARY')
print('=' * 90)
display_df = stats_df[[
    'name', 'hidden_dims', 'depth', 'max_width',
    'batch_norm', 'dropout', 'trainable_params'
]].copy()
display_df['trainable_params'] = display_df['trainable_params'].apply(lambda x: f'{x:,}')
display_df.columns = [
    'Architecture', 'Hidden Dims', 'Depth', 'Max Width',
    'BatchNorm', 'Dropout', 'Trainable Params'
]
print(display_df.to_string(index=False))
print('=' * 90)

# Mark primary
primary = stats_df.loc[stats_df['name'] == 'MLP_Base'].iloc[0]
print(f'\n★ PRIMARY ARCHITECTURE: {primary["name"]}')
print(f'   Hidden dims  : {primary["hidden_dims"]}')
print(f'   Trainable    : {primary["trainable_params"]:,} parameters')
print(f'   BatchNorm    : {primary["batch_norm"]}')
print(f'   Dropout      : {primary["dropout"]}')

# Section 12 — Architecture Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('MLP Architecture Experiment Comparison', fontsize=14, fontweight='bold', y=1.01)

colors = plt.cm.Set2(np.linspace(0, 1, len(stats_df)))
names  = stats_df['name'].tolist()
params = stats_df['trainable_params'].tolist()
depths = stats_df['depth'].tolist()
widths = stats_df['max_width'].tolist()

# ── Plot 1: Parameter counts ──────────────────────────────────────────────────
ax = axes[0]
bars = ax.barh(names, [p/1e3 for p in params], color=colors)
ax.set_xlabel('Trainable Parameters (thousands)')
ax.set_title('Model Size (Params)')
for bar, val in zip(bars, params):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val/1e3:.1f}K', va='center', fontsize=8)
# Highlight primary
primary_idx = names.index('MLP_Base')
bars[primary_idx].set_edgecolor('red')
bars[primary_idx].set_linewidth(2)

# ── Plot 2: Depth vs Width scatter ────────────────────────────────────────────
ax = axes[1]
scatter = ax.scatter(
    depths, widths,
    s=[p/1e2 for p in params],
    c=range(len(names)), cmap='Set2', alpha=0.8
)
for i, (d, w, n) in enumerate(zip(depths, widths, names)):
    ax.annotate(n, (d, w), textcoords='offset points', xytext=(5, 5), fontsize=7)
ax.set_xlabel('Network Depth (hidden layers)')
ax.set_ylabel('Max Width (neurons)')
ax.set_title('Depth vs Width (bubble = params)')

# ── Plot 3: Dropout comparison bar ───────────────────────────────────────────
ax = axes[2]
dropout_vals = stats_df['dropout'].tolist()
bar_colors   = ['#d62728' if n == 'MLP_Base' else c for n, c in zip(names, colors)]
bars2 = ax.bar(names, dropout_vals, color=bar_colors, alpha=0.8)
ax.set_ylabel('Dropout Rate')
ax.set_title('Dropout Regularization per Architecture')
ax.set_xticklabels(names, rotation=35, ha='right', fontsize=8)
ax.set_ylim(0, 0.65)
for bar, val in zip(bars2, dropout_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f'{val:.1f}', ha='center', fontsize=8)

plt.tight_layout()
fig_path = FIGURES_DIR / '01_mlp_architecture_comparison.png'
plt.savefig(fig_path, bbox_inches='tight', dpi=120)
plt.show()
print(f'Figure saved: {fig_path}')

# Section 13 — Architecture Flow Diagram (ASCII & Layer Visualization)

In [ ]:
def draw_mlp_architecture(config: MLPConfig, figsize=(14, 5)):
    """Draw a visual layer-by-layer representation of an MLP architecture."""
    dims = [config.input_dim] + config.hidden_dims + [config.output_dim]
    labels = (['Input'] +
              [f'Hidden {i+1}\n({d})' for i, d in enumerate(config.hidden_dims)] +
              ['Output'])
    regul = (['—'] +
             [f'BN+ReLU\nDO={config.dropout_rate}' if config.use_batch_norm
              else f'ReLU\nDO={config.dropout_rate}'
              for _ in config.hidden_dims] +
             ['Linear\n(no act.)'])

    fig, ax = plt.subplots(figsize=figsize)
    n_layers = len(dims)
    max_dim  = max(dims)
    x_pos    = np.linspace(0.05, 0.95, n_layers)

    cmap = plt.cm.Blues
    for i, (x, dim, label, reg) in enumerate(zip(x_pos, dims, labels, regul)):
        # Normalize bar height
        height = 0.1 + 0.7 * (dim / max_dim)
        y_bot  = 0.5 - height / 2
        color  = cmap(0.3 + 0.6 * (dim / max_dim))
        if i == 0:
            color = '#4CAF50'
        elif i == n_layers - 1:
            color = '#FF5722'

        rect = mpatches.FancyBboxPatch(
            (x - 0.04, y_bot), 0.08, height,
            boxstyle='round,pad=0.01', linewidth=1.5,
            edgecolor='black', facecolor=color, alpha=0.85
        )
        ax.add_patch(rect)
        ax.text(x, 0.5, f'{dim}', ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')
        ax.text(x, y_bot - 0.06, label, ha='center', va='top',
                fontsize=7.5, color='black')
        ax.text(x, 0.5 + height/2 + 0.03, reg, ha='center', va='bottom',
                fontsize=6.5, color='#444', style='italic')

        # Arrow to next layer
        if i < n_layers - 1:
            ax.annotate('', xy=(x_pos[i+1] - 0.045, 0.5),
                        xytext=(x + 0.045, 0.5),
                        arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(
        f'MLP Architecture: {config.name} | '
        f'BN={config.use_batch_norm} | DO={config.dropout_rate} | '
        f'Input({config.input_dim}) → Output({config.output_dim})',
        fontsize=11, fontweight='bold', pad=12
    )
    return fig


# Draw primary architecture
fig = draw_mlp_architecture(base_config, figsize=(14, 5))
fig_path = FIGURES_DIR / '02_mlp_base_architecture_diagram.png'
plt.savefig(fig_path, bbox_inches='tight', dpi=120)
plt.show()
print(f'Architecture diagram saved: {fig_path}')

# Section 14 — Component Ablation: ReLU vs BatchNorm vs Dropout

Measuring the **gradient flow quality** through each architecture at initialization.

In [ ]:
def measure_gradient_flow(model: nn.Module, loader: DataLoader, n_batches: int = 5) -> Dict:
    """
    Run forward+backward passes on n_batches and collect mean absolute
    gradient magnitudes per named parameter group.
    """
    model.train()
    criterion = nn.MSELoss()
    grad_norms = {}

    for step, (batch_x, batch_y) in enumerate(loader):
        if step >= n_batches:
            break
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        preds = model(batch_x)
        loss  = criterion(preds, batch_y)
        loss.backward()

        for name, param in model.named_parameters():
            if param.grad is not None:
                grad_norms.setdefault(name, []).append(
                    param.grad.abs().mean().item()
                )

    model.zero_grad()
    return {k: np.mean(v) for k, v in grad_norms.items()}


# ── Run gradient flow analysis on primary model ───────────────────────────────
print('Measuring gradient flow in MLP_Base at initialization...')
grad_info = measure_gradient_flow(model, train_loader, n_batches=5)

# Aggregate by layer depth
layer_grad_means = {}
for name, val in grad_info.items():
    # Group by block index
    parts = name.split('.')
    if 'hidden_layers' in parts:
        key = f'hidden_{parts[1]}'
    elif 'output_layer' in parts:
        key = 'output'
    else:
        key = 'other'
    layer_grad_means.setdefault(key, []).append(val)

layer_grad_means = {k: np.mean(v) for k, v in layer_grad_means.items()}

print('\nLayer-wise mean |gradient| at initialization:')
for layer, grad_mean in sorted(layer_grad_means.items()):
    bar = '█' * max(1, int(grad_mean * 1000))
    print(f'  {layer:<15}: {grad_mean:.6f}  {bar}')

# ── Visualization ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
layers_sorted = sorted(layer_grad_means.keys())
grad_vals     = [layer_grad_means[k] for k in layers_sorted]

bar_colors = ['#2196F3' if 'hidden' in l else '#FF5722' for l in layers_sorted]
ax.bar(layers_sorted, grad_vals, color=bar_colors, alpha=0.85, edgecolor='black', lw=0.8)
ax.set_xlabel('Layer')
ax.set_ylabel('Mean |Gradient|')
ax.set_title('Gradient Flow at Initialization — MLP_Base (BatchNorm + ReLU)')
ax.set_xticklabels(layers_sorted, rotation=25, ha='right')

plt.tight_layout()
fig_path = FIGURES_DIR / '03_gradient_flow_analysis.png'
plt.savefig(fig_path, bbox_inches='tight', dpi=120)
plt.show()
print(f'Figure saved: {fig_path}')

# Section 15 — Activation Distribution Analysis (Pre vs Post BN)

In [ ]:
def collect_activations(model: nn.Module, batch_x: torch.Tensor) -> Dict[str, np.ndarray]:
    """Attach forward hooks to capture intermediate activations."""
    activations = {}
    hooks = []

    def make_hook(name):
        def hook(module, input, output):
            activations[name] = output.detach().cpu().numpy().flatten()
        return hook

    for i, block in enumerate(model.hidden_layers):
        for j, layer in enumerate(block.block):
            layer_name = type(layer).__name__
            if layer_name in ('ReLU', 'BatchNorm1d'):
                h = layer.register_forward_hook(make_hook(f'block{i}_{layer_name}'))
                hooks.append(h)

    model.eval()
    with torch.no_grad():
        model(batch_x.to(DEVICE))

    for h in hooks:
        h.remove()

    return activations


# ── Collect activations on a single batch ─────────────────────────────────────
sample_x_small = sample_x[:256]
activations    = collect_activations(model, sample_x_small)

# ── Plot activation histograms ────────────────────────────────────────────────
bn_acts  = {k: v for k, v in activations.items() if 'BatchNorm1d' in k}
rel_acts = {k: v for k, v in activations.items() if 'ReLU' in k}

n_blocks = len(model.hidden_layers)
fig, axes = plt.subplots(2, n_blocks, figsize=(4 * n_blocks, 7))
fig.suptitle('Activation Distributions: Post-BatchNorm vs Post-ReLU (MLP_Base)', fontsize=12, fontweight='bold')

for i in range(n_blocks):
    bn_key  = f'block{i}_BatchNorm1d'
    rel_key = f'block{i}_ReLU'

    if bn_key in bn_acts:
        axes[0, i].hist(bn_acts[bn_key], bins=60, color='steelblue', alpha=0.75, density=True)
        axes[0, i].set_title(f'Block {i+1}\nPost-BatchNorm', fontsize=9)
        axes[0, i].set_xlabel('Activation Value')
        axes[0, i].set_ylabel('Density' if i == 0 else '')
        axes[0, i].axvline(0, color='red', lw=1, ls='--', alpha=0.6)
        mean_val = bn_acts[bn_key].mean()
        std_val  = bn_acts[bn_key].std()
        axes[0, i].text(0.05, 0.95, f'μ={mean_val:.3f}\nσ={std_val:.3f}',
                        transform=axes[0, i].transAxes, va='top', fontsize=8,
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    if rel_key in rel_acts:
        axes[1, i].hist(rel_acts[rel_key], bins=60, color='darkorange', alpha=0.75, density=True)
        axes[1, i].set_title(f'Block {i+1}\nPost-ReLU', fontsize=9)
        axes[1, i].set_xlabel('Activation Value')
        axes[1, i].set_ylabel('Density' if i == 0 else '')
        dead_frac = (rel_acts[rel_key] == 0.0).mean()
        axes[1, i].text(0.65, 0.95, f'Dead: {dead_frac*100:.1f}%',
                        transform=axes[1, i].transAxes, va='top', fontsize=8,
                        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.tight_layout()
fig_path = FIGURES_DIR / '04_activation_distributions.png'
plt.savefig(fig_path, bbox_inches='tight', dpi=120)
plt.show()
print(f'Figure saved: {fig_path}')

# Section 16 — Dropout Effect Demonstration

In [ ]:
def mc_dropout_predictions(
    model: nn.Module,
    x: torch.Tensor,
    n_samples: int = 50,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Monte Carlo Dropout: run n_samples forward passes in TRAIN mode
    (dropout active) to estimate predictive uncertainty.

    Returns:
        mean_preds : (N,) mean predictions across MC samples
        std_preds  : (N,) standard deviation (uncertainty estimate)
    """
    model.train()   # keep dropout active
    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            p = model(x.to(DEVICE)).cpu().numpy().flatten()
            preds.append(p)
    model.eval()

    preds_arr  = np.stack(preds, axis=0)   # (n_samples, N)
    mean_preds = preds_arr.mean(axis=0)
    std_preds  = preds_arr.std(axis=0)
    return mean_preds, std_preds


# ── Run MC Dropout on a small batch ──────────────────────────────────────────
demo_x, demo_y = sample_x[:64], sample_y[:64]
mean_p, std_p  = mc_dropout_predictions(model, demo_x, n_samples=50)
true_vals      = demo_y.numpy().flatten()

print('MC Dropout Uncertainty Estimation (50 forward passes, 64 samples):')
print(f'  Mean prediction range  : [{mean_p.min():.2f}, {mean_p.max():.2f}]')
print(f'  Avg uncertainty (std)  : {std_p.mean():.4f}')
print(f'  Max uncertainty (std)  : {std_p.max():.4f}')

# ── Plot uncertainty bands ────────────────────────────────────────────────────
sort_idx = np.argsort(true_vals)
x_axis   = np.arange(len(sort_idx))

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(x_axis,
                mean_p[sort_idx] - 2*std_p[sort_idx],
                mean_p[sort_idx] + 2*std_p[sort_idx],
                alpha=0.3, color='steelblue', label='±2σ MC Dropout Uncertainty')
ax.plot(x_axis, mean_p[sort_idx], color='steelblue', lw=1.5, label='MC Mean Prediction')
ax.scatter(x_axis, true_vals[sort_idx], s=20, color='red', alpha=0.7,
           zorder=5, label='True CCME WQI')
ax.set_xlabel('Sample Index (sorted by true value)')
ax.set_ylabel('CCME WQI')
ax.set_title('MC Dropout Uncertainty Bands — MLP_Base at Initialization (50 passes)')
ax.legend(fontsize=9)
plt.tight_layout()
fig_path = FIGURES_DIR / '05_mc_dropout_uncertainty.png'
plt.savefig(fig_path, bbox_inches='tight', dpi=120)
plt.show()
print(f'Figure saved: {fig_path}')

# Section 17 — Model Factory & Configuration Serialization

In [ ]:
def build_model(config: MLPConfig, device: torch.device = DEVICE) -> WaterQualityMLP:
    """
    Factory function: instantiate a WaterQualityMLP from config and move to device.
    Used by downstream training notebooks (Task 3+) to ensure reproducible setup.
    """
    torch.manual_seed(SEED)
    model = WaterQualityMLP(config).to(device)
    return model


def save_model_checkpoint(
    model     : nn.Module,
    config    : MLPConfig,
    path      : Path,
    extra_meta: Optional[Dict] = None,
) -> None:
    """
    Save a complete model checkpoint:
        - model state dict
        - architecture config (for reconstruction)
        - optional extra metadata (epoch, val_loss, etc.)
    """
    payload = {
        'model_state_dict' : model.state_dict(),
        'architecture_config': config.to_dict(),
        'pytorch_version'  : torch.__version__,
        'seed'             : SEED,
    }
    if extra_meta:
        payload.update(extra_meta)
    torch.save(payload, path)
    print(f'Checkpoint saved → {path}')


def load_model_checkpoint(
    path   : Path,
    device : torch.device = DEVICE,
) -> Tuple[WaterQualityMLP, Dict]:
    """
    Load model from checkpoint.
    Returns the reconstructed model and the full checkpoint dict.
    """
    ckpt   = torch.load(path, map_location=device)
    cfg_d  = ckpt['architecture_config']
    config = MLPConfig(**cfg_d)
    model  = WaterQualityMLP(config).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    return model, ckpt


# ── Save architecture skeleton checkpoint for all variants ───────────────────
print('Saving architecture skeleton checkpoints...')
for cfg_exp in EXPERIMENT_CONFIGS:
    m          = architecture_registry[cfg_exp.name]
    ckpt_path  = MODELS_DIR / f'{cfg_exp.name}_skeleton.pt'
    save_model_checkpoint(m, cfg_exp, ckpt_path, extra_meta={'status': 'skeleton'})

# Save ResidualMLP config separately
resnet_ckpt = {
    'model_state_dict'   : resnet_model.state_dict(),
    'architecture_config': {
        'input_dim': INPUT_DIM, 'hidden_dim': 256, 'num_blocks': 4, 'dropout_rate': 0.2
    },
    'model_type' : 'ResidualMLP',
    'pytorch_version': torch.__version__,
    'status'     : 'skeleton',
}
resnet_path = MODELS_DIR / 'MLP_ResNet_skeleton.pt'
torch.save(resnet_ckpt, resnet_path)
print(f'ResNet checkpoint saved → {resnet_path}')

# ── Verify reload ─────────────────────────────────────────────────────────────
print('\nVerifying checkpoint reload for MLP_Base...')
reloaded_model, reloaded_ckpt = load_model_checkpoint(MODELS_DIR / 'MLP_Base_skeleton.pt')
reloaded_model.eval()
with torch.no_grad():
    reload_out = reloaded_model(sample_x[:32].to(DEVICE))
assert reload_out.shape == (32, 1), 'Reloaded model output shape mismatch'
print('Checkpoint reload verification: PASSED')
print(f'Reloaded config: {reloaded_ckpt["architecture_config"]}')

# Section 18 — Standardized Interface for Task 3 (Training Notebook)

In [ ]:
# ── Confirm all models pass forward-pass verification ────────────────────────
print('Final forward-pass verification for all architectures:')
print(f'{"Architecture":<18} {"Output Shape":<18} {"NaN":<8} {"Inf":<8} {"Status"}')
print('─' * 65)

demo_batch = sample_x[:64].to(DEVICE)

for name, m in architecture_registry.items():
    m.eval()
    with torch.no_grad():
        out = m(demo_batch)
    has_nan = torch.isnan(out).any().item()
    has_inf = torch.isinf(out).any().item()
    status  = '✓ PASS' if (not has_nan and not has_inf) else '✗ FAIL'
    print(f'{name:<18} {str(tuple(out.shape)):<18} {str(has_nan):<8} {str(has_inf):<8} {status}')

print('─' * 65)
print('\n' + '=' * 70)
print('TASK 2 — MLP ARCHITECTURE DEVELOPMENT: COMPLETE')
print('=' * 70)
print(f'  Primary model      : MLP_Base (input 53 → 512 → 256 → 128 → 64 → 1)')
print(f'  Activation fn      : ReLU (inplace)')
print(f'  Normalization      : BatchNorm1d (pre-activation)')
print(f'  Regularization     : Dropout(p=0.30)')
print(f'  Output head        : Linear(64→1) — raw unbounded regression')
print(f'  Weight init        : Kaiming (He) normal, bias=0')
print(f'  Variants defined   : {len(architecture_registry)}')
print(f'  Checkpoints saved  : {MODELS_DIR}')
print(f'  Figures saved      : {FIGURES_DIR}')
print('\nHandoff to Task 3 (Training & Optimization):')
print('  • Use build_model(config) to reconstruct any variant.')
print('  • Use load_model_checkpoint(path) to restore saved skeletons.')
print('  • DataLoaders (train/val/test_loader) are ready with batch_size=1024.')
print('=' * 70)

# Section 19 — Save Architecture Registry Metadata

In [ ]:
# Save complete architecture metadata for downstream notebooks
arch_metadata = {
    'primary_model'    : 'MLP_Base',
    'primary_config'   : base_config.to_dict(),
    'all_architectures': [
        {
            'name'             : cfg_exp.name,
            'hidden_dims'      : cfg_exp.hidden_dims,
            'dropout_rate'     : cfg_exp.dropout_rate,
            'use_batch_norm'   : cfg_exp.use_batch_norm,
            'input_dim'        : cfg_exp.input_dim,
            'output_dim'       : cfg_exp.output_dim,
            'trainable_params' : architecture_registry[cfg_exp.name].count_parameters()['trainable'],
        }
        for cfg_exp in EXPERIMENT_CONFIGS
    ],
    'residual_model': {
        'name'            : 'MLP_ResNet',
        'type'            : 'ResidualMLP',
        'hidden_dim'      : 256,
        'num_blocks'      : 4,
        'dropout_rate'    : 0.2,
        'trainable_params': resnet_model.count_parameters()['trainable'],
    },
    'interface': {
        'input_dim'          : INPUT_DIM,
        'numerical_dim'      : len(NUMERICAL_FEATURES),
        'categorical_ohe_dim': len(ONE_HOT_FEATURES),
        'batch_size'         : BATCH_SIZE,
        'target'             : TARGET,
        'device'             : str(DEVICE),
        'seed'               : SEED,
    }
}

meta_path = PHASE2_DIR / 'configs' / 'architecture_registry.json'
with open(meta_path, 'w') as f:
    json.dump(arch_metadata, f, indent=2)
print(f'Architecture registry saved → {meta_path}')

# Also save stats CSV for Task 5 comparison
stats_csv_path = PHASE2_DIR / 'configs' / 'architecture_stats.csv'
stats_df.to_csv(stats_csv_path, index=False)
print(f'Architecture stats CSV saved → {stats_csv_path}')

print('\n✓ Notebook 02_mlp_architecture.ipynb completed successfully.')